<img src='https://full-stack-assets.s3.eu-west-3.amazonaws.com/Walmart_logo_(2008).svg.png' />

Walmart Inc. est une société multinationale américaine de vente au détail qui exploite une chaîne d'hypermarchés, de grands magasins discount et d'épiceries aux États-Unis, dont le siège social est à Bentonville, Arkansas. L'entreprise a été fondée par Sam Walton en 1962.  

Le service marketing de Walmart nous a demandé de construire un modèle d'apprentissage automatique capable d'estimer les ventes hebdomadaires dans leurs magasins, avec la meilleure précision possible sur les prévisions faites. Un tel modèle les aiderait à mieux comprendre comment les ventes sont influencées par les indicateurs économiques et pourrait être utilisé pour planifier de futures campagnes marketing.

# Import et configuration

In [1]:
import pandas as pd
import numpy as np

import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import  mean_absolute_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, GridSearchCV

# Formatage des nombres après la virgule pour les dataframe pandas
pd.set_option('display.float_format', '{:.3f}'.format)

# Création de la seed pour le machine learning
np.random.seed(42)

# Chargement des données

In [2]:
print('Chargement du dataset...')
df = pd.read_csv('data/Walmart_Store_sales.csv')
print('...Terminé.')

Chargement du dataset...
...Terminé.


# Fonction utilitaire

Définition d'une fonction pour évaluer les performances du modèle.  
Les performances sont directements stockées dans un dictionnaire pour ensuite pouvoir afficher facilement les performances de tous les modèles utilisés dans un dataframe.


In [3]:
resultats = {}
def evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats):
    r2_train = r2_score(Y_train, Y_train_pred)
    r2_test = r2_score(Y_test, Y_test_pred)
    MAE_train = mean_absolute_error(Y_train, Y_train_pred)
    MAE_test = mean_absolute_error(Y_test, Y_test_pred)
    RMSE_train = root_mean_squared_error(Y_train, Y_train_pred)
    RMSE_test = root_mean_squared_error(Y_test, Y_test_pred) 
    MAPE_train = mean_absolute_percentage_error(Y_train, Y_train_pred)
    MAPE_test = mean_absolute_percentage_error(Y_test, Y_test_pred)
    resultats = {
    'r2_train' : round(r2_train,3),
    'r2_test' : round(r2_test,3),
    'MAE_train' : int(round(MAE_train,0)),
    'MAE_test' : int(round(MAE_test,0)),
    'RMSE_train' : int(round(RMSE_train,0)),
    'RMSE_test' : int(round(RMSE_test,0)),
    'MAPE_train' : round(MAPE_train*100,2),
    'MAPE_test' : round(MAPE_test*100,2)
    }  
    return resultats

Définition d'une fonction pour afficher les résultats d'un modèle donné.

In [4]:
def display_results(resultats, model_name):
    res = resultats[model_name]    
    print(f'\nRésultats pour {model_name}')
    print(f'R² (train) : {res['r2_train']:.3f} | R² (test) : {res['r2_test']:.3f}')
    print(f'MAE (train) : {res['MAE_train']} | MAE (test) : {res['MAE_test']}')
    print(f'RMSE (train) : {res['RMSE_train']} | RMSE (test) : {res['RMSE_test']}')
    print(f'MAPE (train) : {res['MAPE_train']:.2f}% | MAPE (test) : {res['MAPE_test']:.2f}%')

# Audit qualité

In [5]:
# Comptage du nombre de lignes et colonnes
print(f'Nombre de lignes : {df.shape[0]}')
print(f'Nombre de colonnes : {df.shape[1]}\n')

print('Affichage du dataset :')
display(df.head())

print('Statistiques basiques :')
data_desc = df.describe(include='all')
display(data_desc)

print('Pourcentage de valeurs manquantes:')
display(100 * df.isnull().sum() / df.shape[0])

missing_values = df.isnull().any().any()
if missing_values :
    print('Il y a des valeurs manquantes dans le dataset')
else :
    print('Aucune valeur manquante dans le dataset')

print('\nVérification du type des données :')
df.info()

duplicates = df.duplicated().sum()
if duplicates > 0 :
    print('\nIl y a des doublons, nettoyage necessaire')
else: 
    print('\nAucun doublon dans le dataset')

Nombre de lignes : 150
Nombre de colonnes : 8

Affichage du dataset :


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.000,18-02-2011,1572117.540,NaN,59.610,3.045,214.778,6.858
1,13.000,25-03-2011,1807545.430,0.000,42.380,3.435,128.616,7.470
2,17.000,27-07-2012,NaN,0.000,NaN,NaN,130.720,5.936
3,11.000,NaN,1244390.030,0.000,84.570,NaN,214.556,7.346
4,6.000,28-05-2010,1644470.660,0.000,78.890,2.759,212.413,7.092


Statistiques basiques :


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000,132,136.000,138.000,132.000,136.000,138.000,135.000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.867,NaN,1249535.828,0.080,61.398,3.321,179.899,7.598
std,6.231,NaN,647463.042,0.272,18.379,0.478,40.275,1.577
min,1.000,NaN,268929.030,0.000,18.790,2.514,126.112,5.143
25%,4.000,NaN,605075.718,0.000,45.587,2.852,131.971,6.598
50%,9.000,NaN,1261423.865,0.000,62.985,3.451,197.909,7.470
75%,15.750,NaN,1806386.200,0.000,76.345,3.706,214.935,8.150


Pourcentage de valeurs manquantes:


Store           0.000
Date           12.000
Weekly_Sales    9.333
Holiday_Flag    8.000
Temperature    12.000
Fuel_Price      9.333
CPI             8.000
Unemployment   10.000
dtype: float64

Il y a des valeurs manquantes dans le dataset

Vérification du type des données :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB

Aucun doublon dans le dataset


# Nettoyage initial des données

Il y a des valeurs manquantes, pas de valeurs aberrantes détectées et aucun doublons. Le nettoyage sera fait après un premier EDA général. 
Pour cette première partie, nous ferons des modifications de confort et pour faciliter la compréhension des données :  
- Transformation en entier des valeurs contenues dans la colonne store 
- Transformation au format date de la colonne Date  
- Modification des valeurs de la colonne temparature pour avoir des températures en °C  

In [6]:
# Transformation de la colonne date
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True).dt.normalize()
df = df.sort_values('Date')

In [7]:
# Modification de la température en °C
df['Temperature'] = (df['Temperature'] - 32) / 1.8

# EDA Brut

**Distribution des colonnes catégorielles**

In [8]:
categorical_col = ['Store', 'Holiday_Flag']

for column in categorical_col: 
    counts = df[column].value_counts().reset_index()
    counts.columns = [column, 'Total']
    counts[column] = counts[column].astype(int)
    counts= counts.sort_values(column, ascending=True)
    cat_dist = px.bar(counts, x=column, y='Total')
    cat_dist.update_xaxes(type='category', categoryorder='array', categoryarray=counts[column])
    cat_dist.update_layout(
        title= dict(text=f'Distribution de la colonne {column}', x=0.5, font_color='blue'), 
        yaxis=dict(title='Nombre d\'observations'),
        showlegend=False, 
        autosize=False, 
        width=1200, 
        height=400
    )
    cat_dist.show()

Les données sont équilibrées entre les magasins, mais les vacances (Holiday) sont rares — ils représentent un signal potentiellement fort mais peu fréquent.

**Distribution des colonnes numériques**

In [9]:
numeric_col = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

cols= 1
rows = (len(numeric_col) + cols - 1) // cols
num_dist = make_subplots(rows=rows, cols=cols, subplot_titles=numeric_col)
for i,column in enumerate(numeric_col): 
    row = i // cols + 1 
    col = i % cols + 1          
    num_dist.add_trace(go.Box(x=df[column], name='', boxpoints='outliers'),row=row, col=col)

num_dist.update_layout(
    title= dict(text='Distribution des colonnes numériques', x=0.5, font_color='blue'), 
    yaxis=dict(title=''), 
    showlegend=False
)
num_dist.show()


Les variables présentent des niveaux de dispersion différents : certaines sont stables (Fuel_Price), d’autres très variables (CPI, Unemployment et Temperature), ce qui suggère des impacts potentiellement différents sur les ventes, notamment non linéaires pour la température et liés au contexte économique pour le chômage et l’inflation.

**Distribution de la cible : ventes hebdomadaires**

In [10]:
target_dist = px.histogram(df, x='Weekly_Sales', nbins=30)
target_dist.update_layout(
    title= dict(text='Distribution des ventes hebdomadaires', x=0.5, font_color='blue'), 
    xaxis=dict(title='Montant des ventes'), 
    yaxis=dict(title='Nombre d\'observations'),
    showlegend=False
)
target_dist.show()

In [11]:
skewness = df['Weekly_Sales'].skew()
print(f'Skewness: {skewness}')

Skewness: 0.08147048584829325


Skewness > 0 : asymétrie à droite  
Comme attendu en analysant la distribution globale des ventes hebdomadaires, une distribution asymétrique à droite ce qui indique beaucoup de petites ventes et quelques très grosses ventes.

# Construction du dataset de modelisation

Supression des lignes avec des valeurs manquantes dans les colonnes Weekly_Sales et Date.  
Weekly_Sales car il s'agit de la cible, il ne doit pas y avoir de valeurs manquantes et Date pour conserver la saisonnalité.

In [12]:
df = df.dropna(subset=['Weekly_Sales','Date'])

Pour le machine learning, la colonne date ne peut pas être utilisée directement par le modèle de machine learning. Elle est donc transformée en variables numériques afin d’extraire l’information temporelle pertinente.    
Création de 4 colonnes distinctes utilisables par le modèle :  
- year : année  
- month : mois  
- dayofyear : jour de l'année  
- week : semaine de l'année  

In [13]:
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['dayofyear'] = df['Date'].dt.dayofyear
df['week'] = df['Date'].dt.isocalendar().week.astype(int)


Selon la consigne imposée, suppression des outliers dans les colonnes numériques : Temperature, Fuel_price, CPI et Unemployment  
Les valeurs considérées comme outliers sont les valeurs qui ne sont pas dans la tranche [moyenne - 3 écart type] et [moyenne + 3 écart type] ([Xˉ−3σ,Xˉ+3σ])

In [14]:
print('Suppression des outliers dans les colonnes numériques...')
to_keep = True
for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    df_mean = df[col].mean()
    df_std = df[col].std()
    to_keep &= (df[col] < df_mean + 3 * df_std) & (df[col] > df_mean - 3 * df_std)

df = df.loc[to_keep, :]
print(f'Terminé. Nombre de lignes restantes : {df.shape[0]}')

Suppression des outliers dans les colonnes numériques...
Terminé. Nombre de lignes restantes : 80


Transformation de la colonne Store en 'int' pour supprimer les virgules et création d'un ordre pour les afficher de 1 à 20 dans les graphiques

In [15]:
df['Store'] = df['Store'].astype(int)
store_order = sorted(df['Store'].unique())

# EDA Général

### Matrice de corrélation entre les valeurs numériques du dataset

In [16]:
corr_matrix = df.corr(numeric_only=True).round(2)

fig_matrix = ff.create_annotated_heatmap(
    corr_matrix.values,
    x = corr_matrix.columns.tolist(),
    y = corr_matrix.index.tolist()
)

fig_matrix.update_layout(
    title= dict(text='Matrice de corrélation', x=0.5, font=dict(size=20, color='blue')),
    width=1600,
    height=800,
    margin=dict(l=150, r=50, t=100, b=150)
    )
fig_matrix.update_xaxes(tickangle=45, tickfont=dict(size=12), side="bottom")
fig_matrix.update_yaxes(tickfont=dict(size=12))
fig_matrix.show()

Le dataset souffre de multicolinéarité dans les variables temporelles imposées : les variables temporelles capturent la même information sous plusieurs formes.  
Les ventes sont principalement influencées par l’inflation (CPI) : les variables économiques ont plus d’impact que les variables météo.

**Analyse des variables externes par rapport aux ventes**

In [17]:
numeric_col = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

cols = 2
rows = (len(numeric_col) + cols - 1) // cols
num_col_sales =make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=numeric_col
)
for i, column in enumerate(numeric_col):
    row = i // cols + 1 
    col = i % cols + 1
    fig = px.scatter(
        df,
        x=column,
        y='Weekly_Sales',
        trendline='ols'
    )
    for trace in fig.data:
        num_col_sales.add_trace(trace, row=row, col=col)

num_col_sales.update_layout(
    title= dict(text=f'Colonnes numériques vs ventes hebdomadaires', x=0.5, font_color='blue'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
    width=1100, 
    height=800
    )
num_col_sales.show()


L'analyse des variables externes par rapport aux ventes hebdomadaires confirme globalement les résultats observés dans la matrice de corrélation :  
- Température vs ventes hedbdomadaires : relation très faible et négative, suggérant que les variations climatiques ont un impact limité sur les ventes.  
- Fuel_Price vs ventes hedbdomadaires : relation quasi nulle, indiquant que le prix du carburant n’influence pas directement les ventes dans ce jeu de données.  
- CPI vs ventes hedbdomadaires : relation la plus prononcées parmi les variables externes, de nature négative, ce qui suggère qu’une hausse de l’inflation est associée à une baisse des ventes.  
- Unemployment vs ventes hedbdomadaires : relation faible et positive, indiquant un impact peu significatif sur les ventes.

**Analyse de la distribution des ventes par magasin**

Les magasins ont-ils des comportements différents ?

In [18]:
box_store_sales = px.box(
    df,
    x='Store',
    y='Weekly_Sales',
    points='outliers'
)
box_store_sales.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=store_order
)
box_store_sales.update_layout(
    title= dict(text='Distribution du total des ventes hebdomadaires par magasin', x=0.5, font_color='blue'), 
    xaxis=dict(title='Store'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
box_store_sales.show()

Ce graphique confirme que la performance des magasins est structurellement différente selon les stores :  
- une forte disparité de performance moyenne  
- une variabilité importante sur certains magasins leaders  
- la présence d’outliers traduisant des pics de ventes ponctuels (probablement liés à des périodes spécifiques comme les promotions ou les fêtes)  

In [19]:
store_sales = df.groupby('Store',observed=False)['Weekly_Sales'].sum().reset_index()
store_sales = store_sales.sort_values(by='Weekly_Sales', ascending=False)
fig_store_sales = px.bar(store_sales, x='Store', y='Weekly_Sales')
fig_store_sales.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=store_sales
)
fig_store_sales.update_layout(
    title= dict(text='Total des ventes hebdomadaires par magasin', x=0.5, font_color='blue'), 
    xaxis=dict(title='Store'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
fig_store_sales.show()

L'analyse du total des ventes hebdomadaires par magasin indique un top 3 des ventes :  
- 1er : store 13  
- 2ème : store 4   
- 3ème : store 1  

Cependant, ce classement est fortement influencé par l’activité globale (volume de données / ancienneté du magasin) et ne reflète pas nécessairement la performance moyenne réelle.

In [20]:
group_mean = df.groupby('Store',observed=False)['Weekly_Sales'].mean().reset_index()
group_mean = group_mean.sort_values(by='Weekly_Sales', ascending=False)
fig_store_sales = px.bar(group_mean, x='Store', y='Weekly_Sales')
fig_store_sales.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=store_sales
)
fig_store_sales.update_layout(
    title= dict(text='Ventes moyennes hebdomadaires par magasin', x=0.5, font_color='blue'), 
    xaxis=dict(title='Store'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
fig_store_sales.show()

En complétant l’analyse par les ventes moyennes hebdomadaires, le classement des magasins évolue significativement par rapport au classement basé sur le volume total :  

- 1er : store 4   
- 2ème : store 14   
- 3ème : store 13  

Ce nouveau classement est plus représentatif de la performance réelle des magasins, car il neutralise l’effet lié à l’ancienneté ou au volume de données disponible par store.  
On observe notamment que le Store 13, bien classé en volume total, reste performant mais est dépassé par les Stores 4 et 14 en moyenne, ce qui suggère une meilleure efficacité commerciale de ces derniers sur une base hebdomadaire.  

**Distribution des ventes en fonction des jours fériés**

Les ventes sont-elles différentes suivants si il y a des vacances ou jours fériés ?

In [21]:
# Ventes moyennes par semaines en fonction des vacances
holiday = df.groupby('Holiday_Flag', observed=False)['Weekly_Sales'].mean().reset_index()
holiday_weekly_sales = px.bar(holiday, x='Holiday_Flag', y='Weekly_Sales')
holiday_weekly_sales.update_layout(
    title= dict(text='Ventes moyennes hebdomadaires selon les vacances', x=0.5, font_color='blue'), 
    xaxis=dict(title='Vacances (0 = non, 1 = oui)'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
holiday_weekly_sales.show()

In [22]:
effect = (
    df[df['Holiday_Flag'] == 1]['Weekly_Sales'].mean()
    / df[df['Holiday_Flag'] == 0]['Weekly_Sales'].mean()
    - 1
) * 100

print(f'Effet des vacances : {effect:.2f} %')

Effet des vacances : 1.31 %


Les ventes pendant les périodes de vacances sont en moyenne 1.31% supérieures aux périodes normales, indiquant un effet positif des vacances sur la performance commerciale.

**Analyse temporelle**

Les ventes sont-elles saisonnières ?

In [23]:
# Ventes moyennes hebdomadaires globales par mois 
group_month = df.groupby('month')['Weekly_Sales'].mean().reset_index()
group_month['month'] = group_month['month'].astype(str)
month_weekly_sales = px.line(group_month, x='month', y='Weekly_Sales')
month_weekly_sales.update_layout(
    title= dict(text='Ventes moyennes hebdomadaires selon le mois', x=0.5, font_color='blue'), 
    xaxis=dict(title='Mois'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
month_weekly_sales.show()

L’analyse des ventes moyennes hebdomadaires par mois met en évidence une forte saisonnalité des ventes. On observe un niveau de ventes globalement plus élevé en décembre, qui constitue un pic très marqué, probablement lié aux périodes de fêtes.  
À l’inverse, les ventes sont plus faibles au milieu de l’année et notamment en octobre et mai, indiquant des périodes de moindre activité commerciale.  
Cette distribution confirme l’importance des effets saisonniers dans l’évolution des ventes hebdomadaires.  

In [24]:
# Ventes moyennes hebdomadaires globales par années
group_year = df.groupby('year')['Weekly_Sales'].mean().reset_index()
group_year['year'] = group_year['year'].astype(int).astype(str)
year_weekly_sales = px.bar(group_year, x='year', y='Weekly_Sales')
year_weekly_sales.update_layout(
    title= dict(text='Ventes moyennes hebdomadaires selon l\'année', x=0.5, font_color='blue'), 
    xaxis=dict(title='Année'), 
    yaxis=dict(title='Montant des ventes hebdomadaires'),
)
year_weekly_sales.show()

Les ventes hebdomadaires évoluent selon une tendance temporelle marquée : elles augmentent entre 2010 et 2011, atteignant un pic en 2011, avant de diminuer en 2012.  
Cela suggère une dynamique non linéaire des ventes dans le temps.

# Machine Learning

## Preprocessing

Séparation de la cible Y des features X

In [25]:
target_name = 'Weekly_Sales'

print('Séparation de la cible des features...')
Y = df.loc[:, target_name]
X = df.drop(columns=['Weekly_Sales', 'Date', 'month', 'dayofyear'], axis=0) 
# Suppression de la colonne date non utlisée en complément de la cible
# Suppresion des colonnes month et dayofyear pour limité la colinéarité
print('...Fait.\n')
print('Cible :')
print(f'{Y.head(3)}\n')
print('Features :')
print(f'{X.head(3)}\n')

print("Division en ensembles d'entraînement et de test...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42) 
print('...Fait.\n')

Séparation de la cible des features...
...Fait.

Cible :
67     461622.220
44    1641957.440
107    994801.400
Name: Weekly_Sales, dtype: float64

Features :
     Store  Holiday_Flag  Temperature  Fuel_Price     CPI  Unemployment  year  \
67       3         0.000        7.617       2.572 214.425         7.368  2010   
44       1         1.000        3.617       2.548 211.242         8.106  2010   
107      8         1.000        0.744       2.548 214.621         6.299  2010   

     week  
67      5  
44      6  
107     6  

Division en ensembles d'entraînement et de test...
...Fait.



### Pipeline : normalisation et encodage

Les variables numériques sont imputées par la médiane afin de réduire l’impact des valeurs extrêmes, tandis que les variables catégorielles sont imputées par la modalité la plus fréquente pour préserver la structure des classes.

Identification des colonnes numériques et catégorielles

In [26]:
categorical_features = ['Store', 'Holiday_Flag']
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'year', 'week']

In [27]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]
)

# Utilisation d'un ColumnTransformer pour créer un objet préprocesseur qui décrit tous les traitements à effectuer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Machine learning du modèle de base : LinearRegression

**Entraînement du modèle**

In [28]:
print('Entraînement du modèle...')
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, Y_train)
print('...Fait.')

Entraînement du modèle...
...Fait.


**Prédictions**

In [29]:
# Prédictions sur le jeu d’entraînement
print('Prédictions sur le jeu d entraînement...')
Y_train_pred = model.predict(X_train)
print('...Fait.')
print(Y_train_pred[0:5])

# Prédictions sur le jeu d’entraînement
print(f'\nPrédictions sur le jeu de test...')
Y_test_pred = model.predict(X_test)
print('...Fait.')
print(Y_test_pred[0:5])

Prédictions sur le jeu d entraînement...
...Fait.
[ 394180.39580661 1874251.48767949 1658957.35158805  487883.96249472
  371981.41741056]

Prédictions sur le jeu de test...
...Fait.
[2057050.19340677  405988.09934968 1647704.01878007 2233266.47139684
  334323.76795625]


c:\Users\grego\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



Ce Warning est classique pour les variables catégorielles à forte cardinalité comme Store, où certaines catégories peuvent être absentes de l’ensemble d’entraînement après split. Une stratification sur cette variable est généralement peu pertinente en raison du déséquilibre des classes.

**Évaluation des performances**

In [30]:
resultats['linear_regression'] = evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats)
display_results(resultats, model_name='linear_regression')


Résultats pour linear_regression
R² (train) : 0.982 | R² (test) : 0.936
MAE (train) : 71779 | MAE (test) : 123528
RMSE (train) : 89151 | RMSE (test) : 176963
MAPE (train) : 7.56% | MAPE (test) : 11.25%


Le modèle de régression linéaire obtient de bonnes performances avec un R² de 0.982 (train) et 0.936 (test), indiquant une bonne capacité prédictive.  
On observe toutefois un léger écart train/test, suggérant une faible perte de généralisation.  
Le MAPE de 11.25% sur le test reste acceptable pour un problème de prévision de ventes.  
Des modèles régularisés comme Ridge et Lasso sont testés pour améliorer la robustesse.  

**Affichage des coeeficients du modèle**

In [31]:
# Récupération des coefficients du modèle et du nom des features associées
lr_features = model.named_steps['preprocessor'].get_feature_names_out().tolist()
lr_coefficients = model.named_steps['regressor'].coef_
# Nettoyage du nom des colonnes
lr_features = pd.Series(lr_features).str.replace('num__','').str.replace('cat__','').str.replace('.0','')
# Création d'un dataframe avec les features et les coefficients
lr_coef_df = pd.DataFrame({'Feature': lr_features,'Coefficient': lr_coefficients})
# Affichage en valeur absolu et triage des cofficients par ordre décroissant
lr_coef_df['abs_coef'] = lr_coef_df['Coefficient'].abs()
lr_coef_df = lr_coef_df.sort_values(by='abs_coef', ascending=False)
# Affichage des coefficients sous forme de graphique
lr_coef = px.bar(
    lr_coef_df, 
    x='Feature', 
    y='abs_coef',
    text='Coefficient', 
    color=lr_coef_df['Coefficient'] > 0, 
    color_discrete_map={
        True: "seagreen",   # impact positif
        False: "orange"  # impact négatif
    })
lr_coef.update_layout(
    title=dict(
        text='Coefficient du modèle de base',
        x=0.5, 
        font_color='blue'
    ),
    xaxis=dict(title= "Variable"),
    yaxis=dict(title= "Importance"),
    width=1400, 
    height=600,
    showlegend=False  
)
lr_coef.update_xaxes(
    categoryorder='array',
    categoryarray=lr_coef_df['Feature']
)
lr_coef.update_traces(
    textposition='outside', 
    cliponaxis=False, 
    textfont_size=10,
    texttemplate='%{text:.0f}'
)
lr_coef.show()

L’analyse des coefficients du modèle de régression linéaire met en évidence que la variable la plus déterminante dans l’explication des ventes est le magasin (Store). En effet, les coefficients associés aux différents stores dominent largement les autres variables, ce qui montre que la performance des ventes dépend avant tout de l’identité et des caractéristiques structurelles de chaque magasin.  

Parmi les facteurs externes, le CPI (inflation) apparaît comme la variable économique la plus influente, confirmant son rôle dans les variations des ventes. À l’inverse, les variables comme la température, le prix du carburant ou les jours fériés ont un impact beaucoup plus limité.  

On observe également que les variables temporelles (année, semaine) et le chômage ont une influence secondaire par rapport aux effets liés aux magasins eux-mêmes.  

## Modèle Lasso

Lasso Regression permet de réaliser une sélection automatique des variables en annulant certains coefficients.  
Entrainement du modèle, prédictions et évaluations de ces performances.

In [32]:
print('Entraînement du modèle...')
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=50000, random_state=42))
])

model.fit(X_train, Y_train)
print('...Fait.')

# Prédictions sur le jeu d’entraînement
print(f'\nPrédictions sur le jeu d entraînement...')
Y_train_pred = model.predict(X_train)
print('...Fait.')

# Prédictions sur le jeu d’entraînement
print(f'\nPrédictions sur le jeu de test...')
Y_test_pred = model.predict(X_test)
print('...Fait.')

resultats['lasso'] = evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats)
display_results(resultats, model_name='lasso')

Entraînement du modèle...
...Fait.

Prédictions sur le jeu d entraînement...
...Fait.

Prédictions sur le jeu de test...
...Fait.

Résultats pour lasso
R² (train) : 0.982 | R² (test) : 0.936
MAE (train) : 71791 | MAE (test) : 123760
RMSE (train) : 89152 | RMSE (test) : 177112
MAPE (train) : 7.55% | MAPE (test) : 11.29%


c:\Users\grego\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



## Modèle Ridge

Ridge Regression permet de réduire l’impact des variables fortement corrélées en pénalisant les coefficients élevés, tout en conservant toutes les variables. 
Entrainement du modèle, prédictions et évaluations de ces performances.

In [33]:
print('Entraînement du modèle...')
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=42))
])

model.fit(X_train, Y_train)
print('...Fait.')

# Prédictions sur le jeu d’entraînement
print(f'\nPrédictions sur le jeu d entraînement...')
Y_train_pred = model.predict(X_train)
print('...Fait.')
print(Y_train_pred[0:5])

# Prédictions sur le jeu d’entraînement
print(f'\nPrédictions sur le jeu de test...')
Y_test_pred = model.predict(X_test)
print('...Fait.')
print(Y_test_pred[0:5])

resultats['ridge'] = evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats)
display_results(resultats, model_name='ridge')

Entraînement du modèle...
...Fait.

Prédictions sur le jeu d entraînement...
...Fait.
[ 623804.53020094 1766980.66597806 1263079.22192984  572444.83721769
  583631.21232224]

Prédictions sur le jeu de test...
...Fait.
[1905111.89706877  614985.0687194  1339220.76067156 1611874.05257971
  583603.57396752]

Résultats pour ridge
R² (train) : 0.910 | R² (test) : 0.870
MAE (train) : 159094 | MAE (test) : 194178
RMSE (train) : 199762 | RMSE (test) : 251658
MAPE (train) : 19.23% | MAPE (test) : 28.85%


c:\Users\grego\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



## Comparaison et interprétation des 3 modèles

In [34]:
df_resultats = pd.DataFrame(resultats).T
display(df_resultats)

,r2_train,r2_test,MAE_train,MAE_test,RMSE_train,RMSE_test,MAPE_train,MAPE_test
linear_regression,0.982,0.936,71779.000,123528.000,89151.000,176963.000,7.560,11.250
lasso,0.982,0.936,71791.000,123760.000,89152.000,177112.000,7.550,11.290
ridge,0.910,0.870,159094.000,194178.000,199762.000,251658.000,19.230,28.850


La régression linéaire simple reste le meilleur compromis entre performance et simplicité. Lasso n’apporte pas d’amélioration notable, tandis que Ridge dégrade les résultats, probablement en raison d’un paramétrage non optimisé.  

La prochaine étape est donc un GridSearchCV, afin d’optimiser les hyperparamètres des modèles régularisés et vérifier s’il est possible d’améliorer la généralisation.

## Optimisation des modèles par Gridsearch

**Optimisation du modèle Lasso**

In [35]:
print('Grid search...')
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(random_state=42))
])

params = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1, 2, 5, 20, 50, 100],
    'regressor__fit_intercept': [True, False],
    'regressor__max_iter' : [20000, 50000, 100000],
'regressor__tol': [1e-4, 1e-3]
}

gridsearch_lasso = GridSearchCV(model, param_grid=params, cv=5, scoring = "neg_root_mean_squared_error", n_jobs=-1) 
gridsearch_lasso.fit(X_train, Y_train)
print('...Fait.')
print(f'Meilleurs paramètres : {gridsearch_lasso.best_params_}')

best_model = gridsearch_lasso.best_estimator_

Y_train_pred = best_model.predict(X_train)
Y_test_pred = best_model.predict(X_test)

resultats['gridsearch_lasso'] = evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats)
display_results(resultats, model_name='gridsearch_lasso')

Grid search...


...Fait.
Meilleurs paramètres : {'regressor__alpha': 100, 'regressor__fit_intercept': True, 'regressor__max_iter': 20000, 'regressor__tol': 0.001}

Résultats pour gridsearch_lasso
R² (train) : 0.982 | R² (test) : 0.936
MAE (train) : 71757 | MAE (test) : 125989
RMSE (train) : 89320 | RMSE (test) : 176777
MAPE (train) : 7.39% | MAPE (test) : 11.68%


c:\Users\grego\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



**Optimisation du modèle Ridge**

In [36]:
print('Grid search...')
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

params = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1, 10],
    'regressor__fit_intercept': [True, False],
    'regressor__solver': ['auto', 'sag', 'sparse_cg']
}

gridsearch_ridge = GridSearchCV(model, param_grid=params, cv=5, scoring = "neg_root_mean_squared_error", n_jobs=-1) 
gridsearch_ridge.fit(X_train, Y_train)
print('...Fait.')
print(f'Meilleurs paramètres : {gridsearch_ridge.best_params_}')

best_model = gridsearch_ridge.best_estimator_

Y_train_pred = best_model.predict(X_train)
Y_test_pred = best_model.predict(X_test)

resultats['gridsearch_ridge'] = evaluate_perf(Y_train, Y_train_pred, Y_test, Y_test_pred, resultats)
display_results(resultats, model_name='gridsearch_ridge')

Grid search...
...Fait.
Meilleurs paramètres : {'regressor__alpha': 0.1, 'regressor__fit_intercept': True, 'regressor__solver': 'sag'}

Résultats pour gridsearch_ridge
R² (train) : 0.978 | R² (test) : 0.949
MAE (train) : 76066 | MAE (test) : 112515
RMSE (train) : 98351 | RMSE (test) : 156994
MAPE (train) : 7.48% | MAPE (test) : 10.57%


c:\Users\grego\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



## Comparaison et interprétation des 3 modèles après optimisation

In [37]:
df_resultats = pd.DataFrame(resultats).T
display(df_resultats)

,r2_train,r2_test,MAE_train,MAE_test,RMSE_train,RMSE_test,MAPE_train,MAPE_test
linear_regression,0.982,0.936,71779.000,123528.000,89151.000,176963.000,7.560,11.250
lasso,0.982,0.936,71791.000,123760.000,89152.000,177112.000,7.550,11.290
ridge,0.910,0.870,159094.000,194178.000,199762.000,251658.000,19.230,28.850
gridsearch_lasso,0.982,0.936,71757.000,125989.000,89320.000,176777.000,7.390,11.680
gridsearch_ridge,0.978,0.949,76066.000,112515.000,98351.000,156994.000,7.480,10.570


Les résultats montrent que les modèles régularisés améliorés par GridSearch surpassent légèrement les modèles initiaux, en particulier pour la généralisation sur le jeu de test.  

- Linear Regression / Lasso (baseline) : très bonnes performances globales, mais légère sur-adaptation et amélioration limitée via Lasso.
- Ridge (baseline) : moins performant, avec une forte dégradation des erreurs, indiquant un mauvais réglage initial des hyperparamètres.
- GridSearch Lasso : performances similaires à la régression linéaire, sans gain significatif.
- GridSearch Ridge : meilleur modèle global, avec un R² test de 0.949 et les meilleures erreurs (MAE, RMSE, MAPE) sur le jeu de test.

**Affichage des coeeficients du meilleur modèle**

In [38]:
# Récupération des coefficients du modèle et du nom des features associées
features = best_model.named_steps['preprocessor'].get_feature_names_out().tolist()
coefficients = best_model.named_steps['regressor'].coef_
# Nettoyage du nom des colonnes
features = pd.Series(features).str.replace('num__','').str.replace('cat__','').str.replace('.0','')
# Création d'un dataframe avec les features et les coefficients
coef_df = pd.DataFrame({'Feature': features,'Coefficient': coefficients})
# Affichage en valeur absolu et triage des cofficients par ordre décroissant
coef_df['abs_coef'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='abs_coef', ascending=False)
# Affichage des coefficients sous forme de graphique
bestmodel_coef = px.bar(
    coef_df, 
    x='Feature', 
    y='abs_coef',
    text='Coefficient', 
    color=coef_df['Coefficient'] > 0, 
    color_discrete_map={
        True: "seagreen",   # impact positif
        False: "orange"  # impact négatif
    })
bestmodel_coef.update_layout(
    title=dict(
        text='Coefficient du meilleur modèle',
        x=0.5, 
        font_color='blue'
    ),
    xaxis=dict(title= "Variable"),
    yaxis=dict(title= "Importance"),
    width=1400, 
    height=600,
    showlegend=False  
)
bestmodel_coef.update_xaxes(
    categoryorder='array',
    categoryarray=coef_df['Feature']
)
bestmodel_coef.update_traces(
    textposition='outside', 
    cliponaxis=False, 
    textfont_size=10,
    texttemplate='%{text:.0f}'
)
bestmodel_coef.show()

Comparé à la régression linéaire classique, le meilleur modèle obtenu via GridSearch Ridge présente une structure d’importance légèrement différente, mais surtout plus régularisée et stable.  

Les variables liées aux stores restent les plus déterminantes. Cependant, on observe une réduction globale de l’amplitude des coefficients, ce qui traduit l’effet de la régularisation Ridge : le modèle limite les extrêmes et réduit l’influence disproportionnée de certains magasins.  

Par ailleurs, les variables externes comme le CPI, le prix du carburant ou la température voient leur importance diminuer encore davantage, indiquant que le modèle privilégie une structure plus conservatrice et moins sensible au bruit.  

# Conclusion

L’analyse exploratoire a mis en évidence une forte hétérogénéité entre magasins, une distribution asymétrique des ventes et une saisonnalité marquée, notamment en fin d’année. Les variables économiques, en particulier le CPI, apparaissent plus influentes que les variables météorologiques.  

L’étude par magasin révèle des différences significatives de performance, aussi bien en volume total qu’en moyenne, ainsi qu’un impact modéré des périodes de vacances et des effets saisonniers mensuels.  

Sur le plan de la modélisation, la régression linéaire offre de bonnes performances de base. Les modèles Ridge et Lasso n’apportent pas d’amélioration initiale, mais leur optimisation via GridSearchCV permet d’obtenir le meilleur compromis.  

Le modèle Ridge optimisé est retenu comme modèle final en raison de ses meilleures performances en généralisation. Il confirme les tendances principales observées, notamment l’importance dominante des stores dans l’explication des ventes, tout en apportant une vision plus stable et régularisée de l’impact des différentes variables.  

# Pour aller plus loin

Plusieurs pistes peuvent être envisagées pour améliorer les performances et la robustesse du modèle.  

- Le feature engineering est le levier principal d’amélioration : création d’interactions entre variables, transformations non linéaires (log, polynômes), ratios pertinents et transformation des variables asymétriques. Ces approches permettent de mieux capturer des relations complexes entre les variables et la cible.  
- D’autres modèles peuvent être testés : ElasticNet (compromis Ridge/Lasso), Random Forest modèle non linéaire robuste aux interactions complexes.  

Pour la mise en production, les modèles robustes à la colinéarité et aux non-linéarités sont à privilégier. Le traitement des outliers via 3-sigma peut être remplacé par des méthodes plus robustes comme l’IQR ou le clipping en contexte réel.  